In [24]:
import pandas as pd; pd.set_option('display.max_columns', 100)
import numpy as np
import optuna

from tqdm import tqdm
from IPython.display import clear_output

import matplotlib.pyplot as plt; plt.style.use('ggplot')
import seaborn as sns
import plotly.express as px
%matplotlib inline

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, RocCurveDisplay, confusion_matrix, accuracy_score, recall_score, precision_score, f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import RFE, RFECV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from precision_recall_cutoff import precision_recall_cutoff

In [ ]:
# reading data file
churn = pd.read_csv('BankChurners.csv', index_col=0)
churn.head()

,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
CLIENTNUM,,,,,,,,,,,,,,,,,,,,,,
768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


# Data Preparation

In [26]:
churn['Attrition_Flag'] = np.where(churn['Attrition_Flag'] == 'Existing Customer', 0, 1)
churn['Gender'] = np.where(churn['Gender'] == 'M', 0, 1)
churn = pd.concat([churn.drop(columns = ['Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category'], axis =  1 ), 
                   pd.get_dummies(churn[['Education_Level', 'Marital_Status', 'Income_Category', 'Card_Category']])], axis = 1)
churn = churn.drop(columns = churn[['Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1', 'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']], axis = 1)
churn.head()

,Attrition_Flag,Customer_Age,Gender,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Education_Level_College,Education_Level_Doctorate,Education_Level_Graduate,Education_Level_High School,Education_Level_Post-Graduate,Education_Level_Uneducated,Education_Level_Unknown,Marital_Status_Divorced,Marital_Status_Married,Marital_Status_Single,Marital_Status_Unknown,Income_Category_$120K +,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown,Card_Category_Blue,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver
CLIENTNUM,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
768805383,0,45,0,3,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False
818770008,0,49,1,5,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False
713982108,0,51,0,3,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False
769911858,0,40,1,4,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False
709106358,0,40,0,3,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False


# Feature engineering

In [27]:
churn['interaction_1'] = churn['Customer_Age'] * churn['Months_on_book'] 
churn['interaction_2'] = churn['Total_Revolving_Bal'] * churn['Avg_Utilization_Ratio'] 
churn['interaction_3'] = churn['Total_Trans_Amt'] * churn['Total_Trans_Ct'] 
churn['interaction_4'] = churn['Avg_Utilization_Ratio'] * churn['Avg_Open_To_Buy'] 
churn['Months_on_book_36_flag'] = np.where(churn['Months_on_book'] == 36, 1, 0)
churn['Credit_Limit_max_flag'] = np.where(churn['Credit_Limit'] == 34516.0, 1, 0)
churn['Total_Revolving_Bal_0_flag'] = np.where(churn['Total_Revolving_Bal'] == 0, 1, 0)
churn['Total_Revolving_Bal_max_flag'] = np.where(churn['Total_Revolving_Bal'] == 2517, 1, 0)
churn['Total_Trans_Ct_under_60'] = np.where(churn['Total_Revolving_Bal'] >= 60, 1, 0)
churn.head()

,Attrition_Flag,Customer_Age,Gender,Dependent_count,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Education_Level_College,Education_Level_Doctorate,Education_Level_Graduate,Education_Level_High School,Education_Level_Post-Graduate,Education_Level_Uneducated,Education_Level_Unknown,Marital_Status_Divorced,Marital_Status_Married,Marital_Status_Single,Marital_Status_Unknown,Income_Category_$120K +,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown,Card_Category_Blue,Card_Category_Gold,Card_Category_Platinum,Card_Category_Silver,interaction_1,interaction_2,interaction_3,interaction_4,Months_on_book_36_flag,Credit_Limit_max_flag,Total_Revolving_Bal_0_flag,Total_Revolving_Bal_max_flag,Total_Trans_Ct_under_60
CLIENTNUM,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
768805383,0,45,0,3,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,1755,47.397,48048,726.754,0,0,0,0,1
818770008,0,49,1,5,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,True,False,False,False,2156,90.720,42603,776.160,0,0,0,0,1
713982108,0,51,0,3,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,False,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,1836,0.000,37740,0.000,1,0,1,0,0
769911858,0,40,1,4,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,True,False,True,False,False,False,1360,1912.920,23420,604.960,0,0,0,1,1
709106358,0,40,0,3,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,False,False,840,0.000,22848,0.000,0,0,1,0,0


# Final Models and Ensemles

In [28]:
## Defining inputs and target
# these are all the variables that showed up at least 90% of the time through all models
x = churn[['Customer_Age', 'Total_Relationship_Count', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'interaction_1', 'interaction_2',
           'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'interaction_3', 'interaction_4', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Avg_Utilization_Ratio']]
y = churn['Attrition_Flag']

In [29]:
gb_accuracy, hgb_accuracy, xgb_accuracy, lgb_accuracy, cat_accuracy = list(), list(), list(), list(), list()
gb_accuracy_scores, hgb_accuracy_scores, xgb_accuracy_scores, lgb_accuracy_scores, cat_accuracy_scores = list(), list(), list(), list(), list()
gb_cv_score, hgb_cv_score, xgb_cv_score, lgb_cv_score, cat_cv_score = list(), list(), list(), list(), list()

## Running 5 times CV
for i in range(5):
    print(i)
    skf = StratifiedKFold(n_splits = 5, random_state = 42, shuffle = True)
    
    for train_ix, test_ix in skf.split(x, y):
        
        ## Splitting the data 
        x_train, x_test = x.iloc[train_ix], x.iloc[test_ix]
        y_train, y_test = y.iloc[train_ix], y.iloc[test_ix]
        
        ## Building gb model ##
        gb_md = GradientBoostingClassifier(n_estimators = 958, max_depth = 3, min_samples_split = 8, min_samples_leaf = 6, learning_rate = 0.05471905209525808).fit(x_train, y_train)
        ## Predicting on X_test and test
        gb_pred = gb_md.predict_proba(x_test)[:, 1]
        ## changing likilyhoods to labels
        gb_label = precision_recall_cutoff(y_test, gb_pred)
        
        ## Building hgb model ##
        hgb_md = HistGradientBoostingClassifier(max_iter = 520, 
                                                learning_rate = 0.1, 
                                                l2_regularization = 0.1, min_samples_leaf = 7, 
                                                max_leaf_nodes = 103, 
                                                max_depth = 3, 
                                                max_bins = 205).fit(x_train, y_train)
        ## Predicting on X_test and test
        hgb_pred = hgb_md.predict_proba(x_test)[:, 1]
        ## changing likilyhoods to labels
        hgb_label = precision_recall_cutoff(y_test, hgb_pred)
        
        ## Building xgb model ##
        xgb_md = XGBClassifier(learning_rate = 0.02, 
                               n_estimators = 622, 
                               max_depth = 6, 
                               min_child_weight = 1, 
                               gamma = 0.0030115414389802427, 
                               alpha = 3.3054988531680833, 
                               colsample_bytree = 0.5111153062068741, 
                               subsample = 0.4337419128945546).fit(x_train, y_train)
        ## Predicting on X_test and test
        xgb_pred = xgb_md.predict_proba(x_test)[:, 1]
        ## changing likilyhoods to labels
        xgb_label = precision_recall_cutoff(y_test, xgb_pred)
        
        ## Building lgb model ##
        lgb_md = LGBMClassifier(n_estimators = 966, 
                                learning_rate = 0.02, 
                                num_leaves = 313, 
                                min_data_in_leaf = 37, 
                                min_child_weight = 0.07766688980407589, 
                                max_depth = 9, 
                                bagging_fraction = 0.6568954564667657, 
                                feature_fraction = 0.5244625947440009, 
                                lambda_l1 = 0.931602725061808, 
                                lambda_l2 = 1.091601491941248).fit(x_train, y_train)
        ## Predicting on X_test and test
        lgb_pred = lgb_md.predict_proba(x_test)[:, 1]
        ## changing likilyhoods to labels
        lgb_label = precision_recall_cutoff(y_test, lgb_pred)
        clear_output()
        
        ## Building cat model ##
        cat_md = CatBoostClassifier(iterations = 699, 
                                    learning_rate = 0.1, 
                                    min_data_in_leaf = 154, 
                                    depth = 7, 
                                    random_strength = 0.7672319235779396, 
                                    bagging_temperature = 0.34078698773221083, 
                                    border_count = 79, 
                                    l2_leaf_reg = 64, 
                                    verbose = False).fit(x_train, y_train)
        ## Predicting on X_test and test
        cat_pred = cat_md.predict_proba(x_test)[:, 1]
        ## changing likilyhoods to labels
        cat_label = precision_recall_cutoff(y_test, cat_pred)
        
        ## Computing metrics
        gb_accuracy.append(accuracy_score(gb_label, y_test))
        hgb_accuracy.append(accuracy_score(hgb_label, y_test))
        xgb_accuracy.append(accuracy_score(xgb_label, y_test))
        lgb_accuracy.append(accuracy_score(lgb_label, y_test))
        cat_accuracy.append(accuracy_score(cat_label, y_test))
        
    gb_accuracy_scores.append(np.mean(gb_accuracy))
    hgb_accuracy_scores.append(np.mean(hgb_accuracy))
    xgb_accuracy_scores.append(np.mean(xgb_accuracy))
    lgb_accuracy_scores.append(np.mean(lgb_accuracy))
    cat_accuracy_scores.append(np.mean(cat_accuracy))

gb_cv_score = np.mean(gb_accuracy_scores)
hgb_cv_score = np.mean(hgb_accuracy_scores)
xgb_cv_score = np.mean(xgb_accuracy_scores)
lgb_cv_score = np.mean(lgb_accuracy_scores)
cat_cv_score = np.mean(cat_accuracy_scores)

print('The accuracy score of the gb model over 5-folds (run 5 times) is:', gb_cv_score)
print('The accuracy score of the hgb model over 5-folds (run 5 times) is:', hgb_cv_score)
print('The accuracy score of the xgb model over 5-folds (run 5 times) is:', xgb_cv_score)
print('The accuracy score of the lgb model over 5-folds (run 5 times) is:', lgb_cv_score)
print('The accuracy score of the cat model over 5-folds (run 5 times) is:', cat_cv_score)

The accuracy score of the gb model over 5-folds (run 5 times) is: 0.9718700049967703
The accuracy score of the hgb model over 5-folds (run 5 times) is: 0.973436973663364
The accuracy score of the xgb model over 5-folds (run 5 times) is: 0.9718574579844734
The accuracy score of the lgb model over 5-folds (run 5 times) is: 0.9728450148075025
The accuracy score of the cat model over 5-folds (run 5 times) is: 0.9738325716305314


In [31]:
from sklearn.ensemble import VotingClassifier

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, stratify = y)

voting_md = VotingClassifier(estimators = [('gb', gb_md), ('hgb', hgb_md), ('xgb', xgb_md), ('lgb', lgb_md), ('cat', cat_md)], voting = 'soft').fit(x_train, y_train)
voting_pred = voting_md.predict_proba(x_test)[:, 1]
voting_label = precision_recall_cutoff(y_test, voting_pred)
    
clear_output() 
print('The accuracy score of the voting model is:', accuracy_score(voting_label, y_test))
print('The recall score of the voting model is:', recall_score(voting_label, y_test))
print('The Roc Auc score of the voting model is:', roc_auc_score(y_test, voting_pred))

The accuracy score of the voting model is: 0.9748272458045409
The recall score of the voting model is: 0.9077380952380952
The Roc Auc score of the voting model is: 0.9932311310089088


In [32]:
import joblib
import pandas as pd

# Assuming you have a fitted model (e.g. best_model, pipeline, etc.)
joblib.dump(voting_md, "model.pkl")

# Also save the exact feature names/order (very important!)
feature_names = x_train.columns.tolist()  # or your final feature list after preprocessing
pd.Series(feature_names).to_csv("feature_names.csv", index=False)

print("Model and features saved!")

Model and features saved!
